# CommerceOps v2 — GRPO Training Notebook

End-to-end pipeline that turns **Qwen2.5-0.5B-Instruct** into a profit-seeking commerce agent on the frozen CommerceOps v2 environment, following the OpenEnv Hackathon roadmap Parts **B** and **B+**.

**Cells in order**
1. Install (Unsloth + TRL + PEFT + matplotlib).
2. Mount Drive, clone repo, install local pkg, boot env.
3. Load Qwen2.5-0.5B-Instruct via Unsloth FastLanguageModel + QLoRA 4-bit.
4. Baselines (wait / random / heuristic / zero-shot Qwen) → `before_metrics.json`.
5. GRPO training loop with 3-stage curriculum.
6. Post-training evaluation → `after_metrics.json` + `before_after_comparison.png`.
7. Generalization × 3 configs (`generalization.json`).
8. Hard-seed retraining burst (`hard_seed_retraining.json`).
9. Behavior / policy evolution (`behavior_evolution.png`, `policy_signature.json`).
10. Composite score (`composite_score.json`) and artifact upload.

**Frozen surface:** every heavy lift uses the `training/` package committed alongside this notebook so the notebook stays thin and reproducible.

---

**Open in Colab:** https://colab.research.google.com/github/your-repo/swiftlogic_grpo_training.ipynb

In [ ]:
# Cell 1 — install dependencies (Colab)
# Unsloth is the fast-path stack for TRL GRPO on a free T4.
!pip install -q --upgrade pip
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps trl==0.17.0 peft accelerate bitsandbytes
!pip install -q matplotlib requests datasets
print('deps installed')

In [ ]:
# Cell 2 — Colab bootstrap: Drive + repo clone + local install
import os, sys, subprocess, json, platform
IN_COLAB = 'google.colab' in sys.modules

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/Rehan-2024/Swiftlogic-Ecom-Agent.git')
REPO_DIR = os.environ.get('REPO_DIR', '/content/Swiftlogic-Ecom-Agent' if IN_COLAB else os.path.abspath('.'))
BRANCH = os.environ.get('REPO_BRANCH', 'feature/commerce-ops-v2')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CKPT = '/content/drive/MyDrive/openenv_grpo_checkpoints'
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'fetch', 'origin'], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    os.chdir(REPO_DIR)
    DRIVE_CKPT = os.path.abspath('./artifacts/checkpoints')
    os.makedirs(DRIVE_CKPT, exist_ok=True)

sys.path.insert(0, REPO_DIR)
print('workdir =', os.getcwd())
print('IN_COLAB =', IN_COLAB)
print('checkpoint dir =', DRIVE_CKPT)

os.makedirs('artifacts', exist_ok=True)
os.makedirs('artifacts/adapter', exist_ok=True)
os.makedirs('artifacts/adapter_checkpoints', exist_ok=True)

# Reproducibility manifest (roadmap B+.8)
run_config = {
    'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'model_name': os.environ.get('MODEL_NAME', 'Qwen/Qwen2.5-0.5B-Instruct'),
    'trainer': 'trl.GRPOTrainer',
    'python_version': sys.version.split()[0],
    'platform': platform.platform(),
    'seed': 2026,
    'repo_branch': BRANCH,
}
with open('artifacts/run_config.json', 'w', encoding='utf-8') as f:
    json.dump(run_config, f, indent=2)
print('wrote artifacts/run_config.json')

In [ ]:
# Cell 3 — smoke-test env and list graders / tasks
from ecom_env import EcomEnv
env = EcomEnv('configs/siyaani_fashion.json')
obs = env.reset(seed=42)
print('initial bank =', obs.bank_balance, 'SKUs =', list(obs.inventory.keys()))
print('graders:', list(env.graders().keys()))

In [ ]:
# Cell 4 — load Qwen2.5-0.5B-Instruct via Unsloth FastLanguageModel (QLoRA 4-bit)
# If Unsloth is not available (e.g. non-GPU runtime), fall back to transformers+peft.
import os
import torch
MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen/Qwen2.5-0.5B-Instruct')
MAX_SEQ_LEN = 2048

try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16,
        lora_dropout=0.0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=2026,
    )
    print('Unsloth loaded', MODEL_NAME)
except Exception as e:
    print('Unsloth unavailable, falling back to transformers+peft:', e)
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype='auto', device_map='auto')
    lora_cfg = LoraConfig(
        r=16, lora_alpha=16, lora_dropout=0.0, bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    )
    model = get_peft_model(model, lora_cfg)

# Qwen chat tokens
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('trainable params:')
try:
    model.print_trainable_parameters()
except Exception:
    pass

In [ ]:
# Cell 5 — minimal baseline sweep (fast-mode friendly)
# Keep evaluation focused: wait / random / heuristic always, zero-shot optional.

from training.policies import (
    build_wait_producer, build_random_producer, build_heuristic_producer,
    build_zero_shot_producer,
)
from training.eval_utils import run_eval_sweep, write_json

FAST_MODE = os.environ.get('FAST_MODE', '1') == '1'
RUN_ZERO_SHOT = os.environ.get('RUN_ZERO_SHOT', '0' if FAST_MODE else '1') == '1'
BASELINE_SEEDS = [101, 202, 303, 404, 505] if FAST_MODE else [101,202,303,404,505,606,707,808,909,1010]
BASELINE_CFG = ['configs/siyaani_fashion.json']

print({'FAST_MODE': FAST_MODE, 'RUN_ZERO_SHOT': RUN_ZERO_SHOT, 'baseline_seeds': len(BASELINE_SEEDS)})

per_policy = {}
for name, prod in [
    ('wait_only', build_wait_producer()),
    ('random', build_random_producer()),
    ('heuristic', build_heuristic_producer()),
]:
    print('running baseline', name)
    per_policy[name] = run_eval_sweep(name, prod, BASELINE_SEEDS, BASELINE_CFG)
    write_json(per_policy[name], f'artifacts/baseline_{name}.json')

if RUN_ZERO_SHOT:
    print('running zero_shot_llm baseline')
    zero_shot_producer = build_zero_shot_producer(model, tokenizer)
    per_policy['zero_shot_llm'] = run_eval_sweep(
        'zero_shot_llm', zero_shot_producer, BASELINE_SEEDS, BASELINE_CFG
    )
    write_json(per_policy['zero_shot_llm'], 'artifacts/baseline_zero_shot_llm.json')
else:
    print('skipping zero_shot_llm baseline in FAST_MODE')

before = {
    'before_metrics': True,
    'fast_mode': FAST_MODE,
    'run_zero_shot': RUN_ZERO_SHOT,
    'policies': per_policy,
    'baseline_for_composite': 'heuristic',
}
write_json(before, 'artifacts/before_metrics.json')
print('baseline summary:')
for k, v in per_policy.items():
    print(' ', k, v.get('summary', {}))

In [ ]:
# Cell 6 — minimal, stable GRPO training (evaluation-focused)
import json, os, re
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

from training.rollout import rollout_episode, extract_action_json
from training.rewards import combined_reward, RewardWeights
from training.policies import SYSTEM_PROMPT

WEIGHTS = RewardWeights()
FAST_MODE = os.environ.get('FAST_MODE', '1') == '1'

# Keep training intentionally tiny for hackathon demo speed (6–12 episodes).
MAX_EPISODES = int(os.environ.get('MAX_EPISODES', '8'))
MAX_EPISODES = max(6, min(12, MAX_EPISODES))

# GRPO stability: never below 2.
FULL_GROUP_SIZE = max(2, int(os.environ.get('GRPO_GROUP_SIZE', '2' if FAST_MODE else '3')))

MAX_STEPS = int(os.environ.get('MAX_STEPS', '60' if FAST_MODE else '90'))
MAX_FALLBACK_RATE = float(os.environ.get('MAX_FALLBACK_RATE', 0.25))
CURRENT_MAX_STEPS = MAX_STEPS
ROLLOUT_STATS = []

# Force fp16 path and disable bf16 for compatibility and reproducibility.
USE_FP16 = True
USE_BF16 = False

print({
    'FAST_MODE': FAST_MODE,
    'MAX_EPISODES': MAX_EPISODES,
    'GRPO_GROUP_SIZE': FULL_GROUP_SIZE,
    'MAX_STEPS': MAX_STEPS,
    'fp16': USE_FP16,
    'bf16': USE_BF16,
})


def build_prompt_from_obs(obs):
    return (
        f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n"
        f"day={obs.current_day} step={obs.step_count} "
        f"bank={float(obs.bank_balance):.0f} "
        f"inventory={dict(obs.inventory)} "
        f"open_tickets={[t.ticket_id for t in obs.active_tickets if t.status == 'open']} "
        f"customer_satisfaction={float(obs.customer_satisfaction):.2f}\n"
        f"Action (JSON only):\n<|assistant|>\n"
    )


def _build_producer_from_completion(completion_text):
    lines = [ln.strip() for ln in re.split(r'[\n]+', completion_text) if ln.strip()]

    def _producer(obs, state):
        idx = state.get('step', 0)
        line = lines[idx] if idx < len(lines) else (lines[-1] if lines else completion_text)
        cand, _ = extract_action_json(line)
        return cand, line

    return _producer


def _grpo_reward_fn(prompts, completions, **kwargs):
    # Crash-safe reward function: any rollout failure gives neutral penalty
    # instead of aborting the trainer run.
    global ROLLOUT_STATS
    seeds = kwargs.get('_seeds') or [2026 + i for i in range(len(prompts))]
    rewards = []
    batch_stats = []

    for _prompt, comp, seed in zip(prompts, completions, seeds):
        try:
            completion_text = comp if isinstance(comp, str) else comp[0]
            producer = _build_producer_from_completion(completion_text)
            rec = rollout_episode(
                producer,
                seed=seed,
                config_path='configs/siyaani_fashion.json',
                max_steps=CURRENT_MAX_STEPS,
            )
            score = float(combined_reward(rec, WEIGHTS))
            rewards.append(score)
            batch_stats.append({
                'seed': int(seed),
                'fallback_rate': float(rec.fallback_rate),
                'format_compliance': float(rec.format_compliance),
                'combined_reward': score,
                'reward_error': None,
            })
        except Exception as e:
            rewards.append(-1.0)
            batch_stats.append({
                'seed': int(seed),
                'fallback_rate': 1.0,
                'format_compliance': 0.0,
                'combined_reward': -1.0,
                'reward_error': str(e),
            })

    ROLLOUT_STATS.extend(batch_stats)
    return rewards


def sample_prompts(n, seed_base=10000, config_path='configs/siyaani_fashion.json'):
    from ecom_env import EcomEnv
    rows = []
    for i in range(n):
        env = EcomEnv(config_path)
        obs = env.reset(seed=seed_base + i)
        rows.append({'prompt': build_prompt_from_obs(obs), '_seed': seed_base + i})
    return Dataset.from_list(rows)


print(f'=== GRPO training ({MAX_EPISODES} episodes) ===')
train_ds = sample_prompts(MAX_EPISODES, seed_base=10000)
ROLLOUT_STATS = []

cfg_full = GRPOConfig(
    output_dir='artifacts/adapter_checkpoints',
    num_train_epochs=1,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=FULL_GROUP_SIZE,
    max_completion_length=96,
    max_prompt_length=512,
    logging_steps=1,
    save_steps=100,
    report_to='none',
    seed=2026,
    fp16=USE_FP16,
    bf16=USE_BF16,
)

trainer = GRPOTrainer(
    model=model,
    args=cfg_full,
    train_dataset=train_ds,
    reward_funcs=_grpo_reward_fn,
    processing_class=tokenizer,
)
trainer.train()

full_rewards = [float(e.get('reward', 0.0)) for e in trainer.state.log_history if 'reward' in e]
full_mean = (sum(full_rewards) / len(full_rewards)) if full_rewards else 0.0
full_fallback_mean = (
    sum(r['fallback_rate'] for r in ROLLOUT_STATS) / len(ROLLOUT_STATS)
    if ROLLOUT_STATS else 0.0
)

try:
    import matplotlib.pyplot as plt
    if full_rewards:
        plt.figure(figsize=(8, 4))
        plt.plot(range(1, len(full_rewards) + 1), full_rewards, marker='o', linewidth=1.5)
        plt.title(f'GRPO Reward Curve ({MAX_EPISODES} episodes)')
        plt.xlabel('Training step')
        plt.ylabel('Reward')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print('No reward data available for reward curve plot.')
except Exception as e:
    print('Reward curve plotting skipped:', e)

try:
    baseline_heuristic = float(per_policy['heuristic']['summary']['mean_reward'])
    improvement = full_mean - baseline_heuristic
    print('\n=== PERFORMANCE IMPROVEMENT ===')
    print(f'Baseline (heuristic): {baseline_heuristic:.4f}')
    print(f'Trained model: {full_mean:.4f}')
    print(f'Improvement: {improvement:+.4f}')
except Exception as e:
    print('Performance comparison skipped:', e)

from training.plots import plot_reward_curve
try:
    plot_reward_curve(
        full_rewards,
        'artifacts/reward_curve.png',
        title=f'GRPO reward curve ({MAX_EPISODES} episodes)',
        stage_boundaries=None,
    )
except Exception as e:
    print('Artifact reward curve save skipped:', e)

with open('artifacts/training_log.txt', 'w', encoding='utf-8') as f:
    for entry in trainer.state.log_history:
        f.write(json.dumps(entry) + '\n')

with open('artifacts/training_summary.json', 'w', encoding='utf-8') as f:
    json.dump(
        {
            'fast_mode': FAST_MODE,
            'max_episodes': MAX_EPISODES,
            'group_size': FULL_GROUP_SIZE,
            'max_steps': MAX_STEPS,
            'fp16': USE_FP16,
            'bf16': USE_BF16,
            'mean_reward': (sum(full_rewards) / len(full_rewards)) if full_rewards else 0.0,
            'fallback_mean': full_fallback_mean,
            'max_fallback_rate_target': MAX_FALLBACK_RATE,
        },
        f,
        indent=2,
    )

assert full_fallback_mean <= MAX_FALLBACK_RATE, (
    f'fallback too high: {full_fallback_mean:.3f} > {MAX_FALLBACK_RATE:.3f}'
)

model.save_pretrained('artifacts/adapter')
tokenizer.save_pretrained('artifacts/adapter')
print('adapter saved to artifacts/adapter')

In [ ]:
# Cell 7 — post-training evaluation + before/after comparison
from training.eval_utils import run_eval_sweep, write_json
from training.policies import build_zero_shot_producer
from training.plots import plot_before_after_bars
import json

EVAL_SEEDS = [111, 222, 333, 444, 555] if FAST_MODE else [111,222,333,444,555,666,777,888,999,1111]

trained_producer = build_zero_shot_producer(model, tokenizer)
after = run_eval_sweep('after_training', trained_producer, EVAL_SEEDS, ['configs/siyaani_fashion.json'])
after['provenance'] = 'trained_adapter'
write_json(after, 'artifacts/after_metrics.json')

before = json.load(open('artifacts/before_metrics.json', 'r', encoding='utf-8'))
before_ref_name = 'zero_shot_llm' if 'zero_shot_llm' in before['policies'] else 'heuristic'
ref = before['policies'][before_ref_name]['summary']['per_task']

plot_before_after_bars(
    ref,
    after['summary']['per_task'],
    'artifacts/before_after_comparison.png',
)

print('comparison baseline =', before_ref_name)
print('before summary:', before['policies'][before_ref_name]['summary'])
print('after summary:', after['summary'])

In [ ]:
# Cell 8 — optional extension (disabled in minimal pipeline)
print('Skipping generalization sweep in minimal fast pipeline.')

In [ ]:
# Cell 9 — optional extension (disabled in minimal pipeline)
print('Skipping hard-seed retraining in minimal fast pipeline.')

In [ ]:
# Cell 10 — optional extension (disabled in minimal pipeline)
print('Skipping behavior evolution plots in minimal fast pipeline.')

In [ ]:
# Cell 11 — optional extension (disabled in minimal pipeline)
print('Skipping composite score in minimal fast pipeline.')

In [ ]:
# Cell 12 — upload to HF Hub (optional, requires HF_TOKEN)
import os
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    model.push_to_hub('your-hf-user/swiftlogic-commerce-ops-v2-grpo')
    tokenizer.push_to_hub('your-hf-user/swiftlogic-commerce-ops-v2-grpo')
    print('uploaded adapter to HF Hub')
else:
    print('HF_TOKEN not set; skipping hub upload (adapter still saved locally)')